# 역할별 메시지와 제약

같은 내용을 사용자 메시지 하나에 넣을 때와 역할별로 나눌 때 응답을 어떻게 비교할 수 있을까요?

## 패키지 준비

첫 실행에서 수업 기준 버전을 설치합니다.

In [1]:
# 현재 커널에 수업 기준 패키지 버전을 설치합니다.
%pip install -q langchain==1.4.0 langchain-openai==1.6.0 python-dotenv==1.2.3

Note: you may need to restart the kernel to use updated packages.


## API 키와 모델 준비

상위 폴더의 `.env`를 찾고 키의 존재만 확인합니다.

In [2]:
# find_dotenv는 실행 폴더부터 상위 폴더의 .env를 찾고 키 값은 출력하지 않습니다.
import json, os, time
from dotenv import find_dotenv, load_dotenv
from langchain.chat_models import init_chat_model

env_path = find_dotenv(usecwd=True)
if env_path:
    load_dotenv(env_path)
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("프로젝트 루트의 .env에 OPENAI_API_KEY가 필요합니다.")
model = init_chat_model("openai:gpt-5.4-mini", max_tokens=220)

## 고정 과제

모든 절에서 같은 고객 문의와 확인 단어를 사용합니다.

In [3]:
# TASK는 모든 비교에서 바꾸지 않는 합성 문의이고 MENTIONS는 품질 점수가 아닙니다.
TASK = """고객 문의: 어제 받은 텀블러 뚜껑이 깨졌습니다. 주문번호는 A-1042입니다.
새 상품으로 바꾸고 싶습니다. 회사 규정상 교환 접수에는 파손 사진이 필요합니다.
답변에는 분류, 필요한 다음 행동, 고객에게 보낼 한 문장을 포함하세요."""
MENTIONS = ["교환", "A-1042", "사진"]

## 측정 함수

단어 포함 표시는 원문 정독을 돕는 표시이며 품질 점수가 아닙니다.

`run(messages)`는 이 노트북에서 작성하는 보조 함수입니다. 역할·내용 목록을 받아 `model.invoke()`를 한 번 실행하고, 답 문자열·사용량·경과 초를 담은 딕셔너리를 반환합니다. `def` 셀은 함수를 정의할 뿐이며 뒤에서 `run(...)`을 호출할 때 모델 요청이 시작됩니다. `mention_flags(text)`는 지정 단어가 문자열에 있는지 표시한 딕셔너리를 반환합니다. 답의 뜻을 판단하는 모델 호출은 포함하지 않습니다.

In [4]:
# 모델 응답의 원문, 사용량, 경과 시간을 한 레코드로 보존합니다.
def run(messages):
    started = time.perf_counter()
    answer = model.invoke(messages)
    return {"text": answer.text, "usage": answer.usage_metadata,
            "latency_s": round(time.perf_counter() - started, 3)}

def mention_flags(text):
    return {word: word in text for word in MENTIONS}

## 사용자 메시지 하나로 실행

업무 규칙과 질문을 한 메시지에 합칩니다.

`run(messages)`는 바로 앞에서 정의한 함수입니다. 메시지 목록을 받아 모델을 한 번 호출하고 `text`, `usage`, `latency_s`를 가진 딕셔너리를 반환합니다. 함수 정의는 요청하지 않으며, 아래 `run(...)` 호출이 실제 실행 시점입니다.

`role_text`는 담당 역할, `constraint_text`는 답변의 제약, `TASK`는 고객 원문입니다. `+ "
" +`는 문장을 줄바꿈으로 이어 붙입니다. 다음 조건은 같은 문구를 서로 다른 메시지 역할에 배치하므로 새로운 지시가 추가된 비교와 구분됩니다.

번역 규칙은 유지하고 번역 대상만 바꾸는 경우에도 같은 구조를 사용할 수 있습니다. 역할을 나누면 교체할 위치가 보이지만 실제 답이 더 나아졌는지는 응답 원문으로 판단해야 합니다.

In [5]:
# 역할 문구와 제약 문구를 분리해 두 조건에 같은 글자를 재사용합니다.
role_text = "당신은 전자상거래 고객 지원 분류기입니다."
constraint_text = "근거 없는 보상은 약속하지 말고 세 항목을 짧게 답하세요."
flat = run([("user", role_text + "\n" + constraint_text + "\n" + TASK)])
print(json.dumps(flat, ensure_ascii=False, indent=2))

{
  "text": "- **분류:** 교환 요청(파손)\n- **필요한 다음 행동:** 파손된 뚜껑 사진을 요청한 뒤, 규정에 따라 교환 접수를 진행하세요.\n- **고객에게 보낼 한 문장:** 주문번호 A-1042 확인했습니다; 교환 접수를 위해 파손된 뚜껑 사진을 보내주시면 확인 후 다음 절차를 안내해 드리겠습니다.",
  "usage": {
    "input_tokens": 117,
    "output_tokens": 98,
    "total_tokens": 215,
    "input_token_details": {
      "audio": 0,
      "cache_read": 0
    },
    "output_token_details": {
      "audio": 0,
      "reasoning": 0
    }
  },
  "latency_s": 1.354
}


## 역할별 메시지로 실행

공통 행동은 system, 과업 제약은 developer, 사례는 user에 둡니다. developer 역할은 제공자별 지원 차이를 확인해야 합니다.

In [6]:
# 같은 두 문구를 system과 developer로 나누고 고객 문의는 user에 둡니다.
layered = run([
    ("system", role_text),
    ("developer", constraint_text),
    ("user", TASK),
])
print(json.dumps(layered, ensure_ascii=False, indent=2))

{
  "text": "- **분류:** 파손/불량 교환 요청  \n- **필요한 다음 행동:** 파손된 뚜껑 사진을 접수해 주세요. 주문번호 **A-1042**로 확인 후 교환 가능 여부를 안내합니다.  \n- **고객에게 보낼 한 문장:** 파손 부위가 보이도록 사진을 보내주시면 주문번호 A-1042 기준으로 교환 접수를 도와드리겠습니다.",
  "usage": {
    "input_tokens": 125,
    "output_tokens": 102,
    "total_tokens": 227,
    "input_token_details": {
      "audio": 0,
      "cache_read": 0
    },
    "output_token_details": {
      "audio": 0,
      "reasoning": 0
    }
  },
  "latency_s": 1.467
}


## 고정 루브릭으로 비교

문장 취향이 아니라 미리 정한 세 항목으로 두 원문을 확인합니다.

In [7]:
# 두 응답의 단어 표시, 사용량, 지연 시간을 조건 이름별 딕셔너리로 묶습니다.
comparison = {
    "user_only": {**mention_flags(flat["text"]), "usage": flat["usage"], "latency_s": flat["latency_s"]},
    "role_split": {**mention_flags(layered["text"]), "usage": layered["usage"], "latency_s": layered["latency_s"]},
}
print(json.dumps(comparison, ensure_ascii=False, indent=2))

{
  "user_only": {
    "교환": true,
    "A-1042": true,
    "사진": true,
    "usage": {
      "input_tokens": 117,
      "output_tokens": 98,
      "total_tokens": 215,
      "input_token_details": {
        "audio": 0,
        "cache_read": 0
      },
      "output_token_details": {
        "audio": 0,
        "reasoning": 0
      }
    },
    "latency_s": 1.354
  },
  "role_split": {
    "교환": true,
    "A-1042": true,
    "사진": true,
    "usage": {
      "input_tokens": 125,
      "output_tokens": 102,
      "total_tokens": 227,
      "input_token_details": {
        "audio": 0,
        "cache_read": 0
      },
      "output_token_details": {
        "audio": 0,
        "reasoning": 0
      }
    },
    "latency_s": 1.467
  }
}
